# Agentic Financial Research - end-to-end demo

Two agents with restricted tool sets research one ticker through a **typed
state machine** (LangGraph), exchange a serialized brief and a clarification
cycle, and produce a validated 3-risk report whose numbers are computed in
Python. Design: `docs/plan_task3_agentic.md`.

| Part | What it proves |
|---|---|
| 0 | env wiring, offline imports |
| 1 | single-agent mode: one agent, all five tools, persisted artefacts |
| 2 | tracing: narrative + summary counts from the JSONL trace |
| 3 | memory follow-up: answers WITHOUT tool calls (counter unchanged) |
| 4 | two-agent mode: brief->critique->clarification loop, exactly one cycle |
| 5 | cache HIT rerun: second run replays the day's cache, zero tool calls |
| 6 | fault injection: a forced tool failure produces a stamped replan |
| 7 | dashboard hint (run locally) |

Live gateway runs cost real time (thinking models are slow) and rate limits.
The unit suite (`pytest agentic_research/tests`, 56 tests) runs fully offline.


## Part 0 - setup

In [1]:
import json
import sys
from pathlib import Path

cwd = Path.cwd()
candidates = [cwd] + list(cwd.parents)
PROJECT = next(c / "agentic_research" for c in candidates
               if (c / "agentic_research" / "src" / "main.py").exists())
sys.path.insert(0, str(PROJECT))

print("project:", PROJECT)
from src import config  # noqa: E402  (verifies the import path)

print("trace file:", config.LOG_TRACE_JSONL.name)

project: C:\MIHA\CDAZZDEV\agentic_research
trace file: agent_trace.jsonl


In [2]:
from src.llm_client import load_env  # noqa: E402

load_env()
import os  # noqa: E402

key = os.environ.get("LLM_API_KEY", "")
assert key, "LLM_API_KEY missing - copy agentic_research/.env.example to the repo-root .env"
print("gateway:", os.environ.get("LLM_BASE_URL", config.DEFAULT_BASE_URL))
print("model  :", os.environ.get("LLM_MODEL", config.DEFAULT_MODEL))
print("key    :", key[:6] + "...*masked*")
config.ensure_dirs()

gateway: https://api.commandcode.ai/provider/v1
model  : z-ai/glm-5.3-flash
key    : user_2...*masked*


## Part 1 - single-agent mode

One agent with all five tools works the objective autonomously. The final
report is validated by Pydantic (three risks, evidence-tagged) and saved to
`outputs/report.json` + `outputs/report.md`. Expect several minutes on a live
thinking gateway.


In [3]:
from src import tools  # noqa: E402
from src.main import client_from_env  # noqa: E402

demo_client = client_from_env(use_cache=True)
demo_ticker = "NVDA"

price_res = tools.get_price_data(demo_ticker, "1y")
news_res = tools.get_news(demo_ticker, 5)
vol_res = tools.calculate_volatility(demo_ticker, 90)
sent_res = tools.llm_sentiment(
    ["NVDA raises guidance on AI demand", "Chip sector slips after earnings"],
    demo_client, ticker=demo_ticker)
search_res = tools.web_search("NVDA analyst commentary outlook")

for name, res in [("get_price_data", price_res), ("get_news", news_res),
                  ("calculate_volatility", vol_res), ("llm_sentiment", sent_res),
                  ("web_search", search_res)]:
    payload = res.data if res.ok else res.error
    print(f"{name:22s} ok={res.ok} source={res.source:12s} type={type(res).__name__}")
    print("   ", str(payload)[:150])

assert price_res.ok and price_res.data["current_price"] > 0
assert isinstance(news_res.data["headlines"], list)
assert vol_res.data["band"] in {"low", "moderate", "high"}
assert sent_res.data["label"] in {"positive", "negative", "neutral"}
print("all five tools returned correctly typed ToolResult payloads")

C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


get_price_data         ok=True source=yfinance     type=ToolResult
    {'ticker': 'NVDA', 'period': '1y', 'current_price': 239.24000549316406, 'previous_close': 238.89999389648438, 'week52_high': 243.3699951171875, 'week5
get_news               ok=True source=news-ladder  type=ToolResult
    {'headlines': [{'title': 'Why the traditional 60/40 portfolio might be dead', 'source': 'Yahoo Finance RSS', 'published': '2026-10-07', 'url': 'https:
calculate_volatility   ok=True source=yfinance     type=ToolResult
    {'ticker': 'NVDA', 'annualized': 0.3932, 'daily': 0.024771, 'window': 90, 'n_obs': 90, 'band': 'moderate'}
llm_sentiment          ok=True source=llm          type=ToolResult
    {'items': [{'headline': 'NVDA raises guidance on AI demand', 'sentiment': 'positive', 'confidence': 0.95, 'brief_reason': "Raising guidance signals be
web_search             ok=False source=duckduckgo   type=ToolResult
    no search results
all five tools returned correctly typed ToolResult payloads


In [4]:
from src.main import run_single_agent_mode

TICKER = "NVDA"
single_result = run_single_agent_mode(TICKER, use_cache=True)
print("run id:", single_result["run_id"])
print("tool call counter:", single_result["store"].tool_call_count)

C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


single hit the step cap (8)


run id: 6e1e5823f969
tool call counter: 8


In [5]:
from IPython.display import Markdown, display  # noqa: E402

display(Markdown((PROJECT / "outputs" / "report.md").read_text(encoding="utf-8")))

# NVDA — agentic research report

## Financial health summary

The quantitative brief was unavailable, so fundamental health (revenue, margins, leverage, cash position) could not be verified and no such figures are asserted here. Qualitatively, NVIDIA presents as the established leader in accelerated computing — its own materials describe GPUs driving AI, HPC, gaming, autonomous vehicles, and robotics — with a product cadence stretching back to the Ampere/A100 generation announced May 14, 2020. The measurable risk picture is dominated by volatility: the Python computation puts annualized_vol at 0.3932, implying a 90-day 1-sigma band of 183.02–295.46 around the 239.24 spot, a wide envelope that signals material two-way price risk regardless of fundamentals. Market context adds caution: the index rally is record-setting but narrowing in breadth, and a 5.3% 10-year Treasury yield sets a high hurdle for growth-oriented exposure.

Evidence: quant_brief (unavailable — noted, no fundamentals asserted), market_commentary: nvidia.com (GPU/AI/HPC/gaming/autonomous vehicles/robotics positioning), market_commentary: Wikipedia (Ampere/A100 announced May 14, 2020), python_vol_band: price=239.24, annualized_vol=0.3932, lower_1sd=183.02, upper_1sd=295.46, headlines: Yahoo Finance RSS 2026-10-07 (narrowing breadth; 10-Year Treasury at 5.3%)

## Top three 90-day risks

1. **Measured volatility is high: the 90-day 1-sigma band spans 183.02–295.46** (likelihood 4) - The Python volatility computation puts annualized_vol at 0.3932, implying an expected 1-sigma move of ±56.22 over 90 days — a band of 183.02 to 295.46 around spot 239.24. That width is the primary, measured risk: even absent any company-specific event, one-standard-deviation price travel over a quarter is large. The 2026-10-07 headline that 'the breadth of the rally has narrowed' identifies a plausible catalyst for testing the lower half of that band — when index gains concentrate in fewer names, any rotation away from the leaders hits mega-cap AI names like NVIDIA first. Treat 183.02 as the reference level for downside planning and 295.46 as the modeled 1-sigma upside.
   evidence: python_vol_band: annualized_vol=0.3932; expected_1sd_move_90d=56.22; lower_1sd=183.02; upper_1sd=295.46; price=239.24; horizon_days=90; headlines (Yahoo Finance RSS): 'S&P 500, Nasdaq hit records — but the breadth of the rally has narrowed' (2026-10-07)
2. **Narrow rally leadership plus institutional exits from AI-chip peers** (likelihood 3) - Two 2026-10-07 items point in the same direction: the S&P 500 and Nasdaq hit records 'but the breadth of the rally has narrowed,' and Warren Buffett's $863 million 'secret' portfolio dumped Alphabet and Broadcom — Broadcom being a direct AI-semiconductor peer of NVIDIA. Together they describe a market where gains depend on a shrinking set of leaders while at least one prominent institutional holder has demonstrated willingness to exit crowded big-tech/AI-chip positions. NVIDIA sits at the center of that same crowded trade (its own positioning is as the leader in AI computing), so a leadership rotation or de-rating of AI semis would transmit to it directly. This is a positioning/flow risk rather than a company-fundamental one — notably, no NVDA-specific headlines appeared in the feed.
   evidence: headlines (Yahoo Finance RSS): 'S&P 500, Nasdaq hit records — but the breadth of the rally has narrowed' (2026-10-07); headlines (Yahoo Finance RSS): Warren Buffett's $863 million 'secret' portfolio dumped Alphabet and Broadcom (2026-10-07); market_commentary (web): nvidia.com self-description: 'World Leader in Artificial Intelligence Computing' — GPUs driving AI, HPC, gaming, autonomous vehicles, robotics
3. **5.3% 10-year yield raises the discount-rate hurdle for growth exposure** (likelihood 3) - The 10-year Treasury yields 5.3% versus 3.4% for Schwab's dividend ETF (2026-10-07): risk-free income now out-yields diversified equity income, a high discount-rate environment that is structurally unkind to growth stocks whose value rests on distant cash flows — a category NVIDIA typifies given its AI-computing focus. The companion piece questioning whether the traditional 60/40 portfolio is dead compounds the problem: if bonds no longer reliably cushion equity drawdowns, the case for holding high-volatility equity (annualized_vol 0.3932 per the Python computation) weakens unless hedged. Limit noted: with the quantitative brief unavailable, NVIDIA's own valuation sensitivity to rates could not be quantified from the bundle — the macro datum stands on its own.
   evidence: headlines (Yahoo Finance RSS): '10-Year Treasuries Yield 5.3%; Schwab's Dividend ETF Yields 3.4%' (2026-10-07); headlines (Yahoo Finance RSS): 'Why the traditional 60/40 portfolio might be dead' (2026-10-07); python_vol_band: annualized_vol=0.3932 — elevated realized volatility raises the cost of unhedged growth exposure in a high-rate environment

## Hedge strategy

90-day collar built directly on the Python-computed 1-sigma band: long a 90-day put struck at the lower bound (183.02) financed in part by a short 90-day call struck at the upper bound (295.46).

The Python computation defines the 1-sigma 90-day envelope as 183.02–295.46 around spot 239.24 (expected 1-sigma move ±56.22). Placing the put strike at 183.02 insures precisely the modeled 1-sigma downside, while the short 295.46 call caps upside at the modeled 1-sigma high — the collar converts the statistical band into a hard P&L boundary over the same 90-day horizon the band was computed on. Sizing is relative to the band by construction: strikes are set per share at the band bounds, and the number of contracts is scaled to the shares held so coverage matches the position. Given annualized_vol of 0.3932, 90-day options should carry meaningful premium, which is why the short call is included; if retaining full upside beyond 295.46 matters more than cost, run the 183.02 put outright instead.

Instruments: Long 90-day put option, strike 183.02 (lower 1-sigma bound of the Python band), Short 90-day call option, strike 295.46 (upper 1-sigma bound of the Python band), to defray the put premium, Contract count scaled to shares held using standard exchange option-lot sizing, so the per-share strikes coincide exactly with the band bounds
Data basis: python_vol_band: price=239.24, lower_1sd=183.02, upper_1sd=295.46, expected_1sd_move_90d=56.22, horizon_days=90 — collar strikes set at the band bounds over the same 90-day horizon; python_vol_band: annualized_vol=0.3932 — elevated realized volatility supports an options-based hedge and implies non-trivial 90-day option premium; headlines (Yahoo Finance RSS, 2026-10-07): 'the breadth of the rally has narrowed' — hedging into a narrow, record-setting rally where rotation risk is elevated
Cost note: The collar is premium-reducing, not necessarily zero-cost: the short 295.46 call offsets the long 183.02 put, but the net debit depends on implied volatility at execution, which the bundle does not supply (only realized annualized_vol 0.3932). Buying the 183.02 put outright preserves all upside above 295.46 but requires paying the full premium. Either structure should be sized to the shares held at standard option-lot sizing and priced at execution.

---

This report is generated automatically for informational purposes only and is not investment advice.


## Part 2 - tracing

The trace is a raw JSONL you can diff and grep. `printing` renders it as a
narrative with per-agent counts.


In [6]:
from src import printing  # noqa: E402

entries = printing.load_trace(run_id=single_result["run_id"])
print(printing.summarize(entries))
print()
print("narrative (first 6000 chars):")
print(printing.render_narrative(entries)[:6000])
assert any("REPLAN" in line for line in printing.render_narrative(entries).splitlines()) \
    or printing.render_narrative(entries).count("handoff") >= 1
print()
print("the observe -> decide loop is fully recorded in agent_trace.jsonl")

trace summary:
  tool calls per agent: {'single': 8}
  replans: 2 | critique events: 0 | memory events: 0 | cache events: 0
  failed tool calls: 0

narrative (first 6000 chars):
[single] decide: next_action
[single] get_price_data({"ticker": "NVDA", "period": "1y"}) -> ok
    observe: {"ok": true, "data": {"ticker": "NVDA", "period": "1y", "current_price": 239.24000549316406, "previous_close": 238.89999389648438, "week52_high": 243.3699951171
[single] decide: next_action
[single] get_news({"ticker": "NVDA", "n": 10}) -> ok
    observe: {"ok": true, "data": {"headlines": [{"title": "Why the traditional 60/40 portfolio might be dead", "source": "Yahoo Finance RSS", "published": "2026-10-07", "ur
[single] decide: next_action
[single] llm_sentiment({"headlines": ["Why the traditional 60/40 portfolio might be dead", "S&P 500, Nasdaq hit records — but 'the breadth of the rally", "10-Year Treasuries Yield 5.3%; Schwab's Dividend ETF Yields"]}) -> ok
    observe: {"ok": true, "data": {"items":

## Part 3 - memory follow-up

The follow-up is answered from the session record ONLY: no tool calls, so the
tool-call counter cannot move (asserted below).


In [7]:
from src.main import answer_from_memory  # noqa: E402

outcome = answer_from_memory(
    "According to the session record, what was NVDA's latest close price, and "
    "is it above or below the 50- and 200-day SMA?", use_cache=True)
assert outcome["counter_unchanged"], "memory answers must not run tools"
print(outcome["answer"])
print("tool calls before/after:", outcome["tool_call_count_before"], "->",
      outcome["tool_call_count_after"])

According to the session record, NVDA's latest close was $239.24 (2026-10-06 row; the record also lists current_price of ~239.24 and previous_close of ~238.90). NVDA is above both the 50-day SMA (sma50_stance: 'above') and the 200-day SMA (sma200_stance: 'above').
tool calls before/after: 8 -> 8


## Part 4 - two-agent mode (the main graph)

`run_research` compiles the LangGraph state machine:
cache_check -> agent_a_brief (quant) -> agent_b_gather_and_critique (news,
commentary) -> agent_a_respond (sentiment + 30d vol) -> agent_b_final_report
-> save_cache, with the critique cycle capped at exactly one.


In [8]:
import os  # noqa: E402

from datetime import date  # noqa: E402

from src import config  # noqa: E402
from src.graph import run_research  # noqa: E402

TICKER = "NVDA"

# demo-only: make THIS cell show a fresh graph run although earlier live runs
# already persisted today's brief; a real second run on the same day simply
# loads that file instead (next part proves it)
today_cache = config.CACHE_DIR / f"{TICKER}_{date.today().isoformat()}.json"
if today_cache.exists():
    today_cache.unlink()

research = run_research(TICKER, use_cache=True)
print("cache_hit:", research["cache_hit"], "| degraded:", research["degraded"],
      "| error:", research["error"][:60])
report = research["report"]
print("meta:", json.dumps(report["meta"], indent=1))

C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


agent_b hit the step cap (6)


cache_hit: False | degraded: False | error: 
meta: {
 "run_id": "c8a6eb71fd09",
 "run_at": "2026-10-07T10:35:52+00:00",
 "tool_call_count": 12,
 "replans": 6,
 "critique_cycles": 0,
 "memory_hits": 0,
 "degraded": false
}


In [9]:
from src.schemas import ResearchReport

validated = ResearchReport.model_validate(research["report"])
print("verified: exactly three risks, likelihood in 1..5, every risk carries evidence")
for risk in validated.risks:
    evidence = risk.evidence[0]
    print(f"  [{risk.likelihood}/5] {risk.title}")
    print(f"      evidence: {evidence.source_tool}: {evidence.key_datum[:70]}")

verified: exactly three risks, likelihood in 1..5, every risk carries evidence
  [4/5] Overbought technicals just below the 52-week high
      evidence: get_price_data: price 239.24 vs 52-week high 243.37; %B 0.9675; RSI 67.16; above SMA50
  [3/5] Discount-rate pressure from elevated Treasury yields
      evidence: llm_sentiment: negative item: '10-Year Treasuries Yield 5.3%' - raises discount rates
  [3/5] Narrowing rally breadth concentrates fragility in mega-cap leaders
      evidence: llm_sentiment: headline 'S&P 500, Nasdaq hit records - but the breadth of the rally h


In [10]:
print(printing.render_file(run_id=research["run_id"])[:3000])

[cache] no cache for today - full run
[agent_a] decide: next_action
[agent_a] get_price_data({"ticker": "NVDA", "period": "1d"}) -> ok
    observe: {"ok": false, "data": null, "error": "period '1d' not allowed", "hint": "period must be one of 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max", "source": "yfinance", 
[agent_a] decide: next_action
[agent_a] REPLAN: Previous call failed because period '1d' is not allowed; switching to the shortest valid period '1mo' to still obtain recent daily data. (get_price_data|{"period": "1d", "ticker": "NVDA"} -> get_price_data)
[agent_a] get_price_data({"ticker": "NVDA", "period": "1mo"}) -> ok
    observe: {"ok": true, "data": {"ticker": "NVDA", "period": "1mo", "current_price": 239.24000549316406, "previous_close": 238.89999389648438, "week52_high": 243.369995117
[agent_a] decide: next_action
[agent_a] REPLAN: Indicator fields returned unknown/None, indicating the previous period lacked enough history; switching to 1y so SMA50/SMA200/RSI/MACD/%B can be c

### 4b - what the agents said and handed off

The brief that Agent A handed to Agent B (the *typed* `AgentBrief`), and the
tool calls each agent recorded, straight from the session store. The critique
request and the clarification that closed it are part B of this trace below
the graph's printed narrative.

In [11]:
store = research["store"]

print("AGENT A -> AGENT B brief_v2 (typed AgentBrief as JSON):")
print(json.dumps(store.briefs["brief_v2"], indent=1)[:1500])
calls = sorted(f'{e.split("|", 1)[0]}:{v["tool"]}'
               for e, v in store.tool_results.items()
               if v.get("tool"))
print()
print("tool calls recorded in memory:", calls)

AGENT A -> AGENT B brief_v2 (typed AgentBrief as JSON):
{
 "ticker": "NVDA",
 "as_of": "2026-10-07T10:25:43+00:00",
 "price_level": {
  "current": 239.24000549316406,
  "previous_close": 238.89999389648438,
  "week52_high": 243.3699951171875,
  "week52_low": 164.27000427246094,
  "ytd_pct": 28.28
 },
 "volatility": {
  "annualized_90d": 0.3932,
  "annualized_30d": 0.3686,
  "daily": 0.024771,
  "band": "moderate"
 },
 "indicator_state": {
  "sma50_stance": "above",
  "sma200_stance": "above",
  "rsi": 67.16310915228561,
  "macd_hist": 1.379679454797433,
  "macd_fresh_cross": false,
  "pct_b": 0.9675410212539503
 },
 "sentiment": {
  "overall_score": 0.0278,
  "label": "neutral",
  "counts": {
   "positive": 2,
   "negative": 1,
   "neutral": 7
  },
  "items": [
   {
    "headline": "Why the traditional 60/40 portfolio might be dead",
    "sentiment": "neutral",
    "confidence": 0.65,
    "brief_reason": "General portfolio-allocation commentary with no NVDA-specific catalyst, so no cle

## Part 5 - cache HIT rerun

The same ticker on the same calendar day replays the persisted brief/report
with ZERO new tool calls.


In [12]:
rerun = run_research(TICKER, use_cache=True)
assert rerun["cache_hit"] is True and rerun["report"] is None, "expected a cache HIT"
payload = rerun["cache_payload"]
print(f"cache HIT - loaded brief for {payload['ticker']} on {payload['as_of']}")
print(f"tool calls this run: 0 (everything came from {payload['ticker']}_"
      f"{payload['as_of']}.json)")
print("counters stayed:", json.dumps(payload["meta"]))

cache HIT - loaded brief for NVDA on 2026-10-07
tool calls this run: 0 (everything came from NVDA_2026-10-07.json)
counters stayed: {"run_id": "c8a6eb71fd09", "tool_call_count": 12, "replans": 6, "critique_cycles": 1}


## Part 5b - whitelist enforcement at runtime

Agent B tries a price tool and the DISPATCHER refuses it *inside* the loop;
the refusal becomes a normal observation and B continues with its own tools.
This is enforcement, not prompt etiquette - see the `REFUSED` trace event.

In [13]:
from src import memory as memory_pkg  # noqa: E402
from src.schemas import AgentAction  # noqa: E402
from src.agents import AgentLoop, CrossAgentToolAccessError, demo_blocked_access  # noqa: E402
from src.llm_client import client_from_env  # noqa: E402
from src.memory import MemoryStore  # noqa: E402
from src.tracing import TraceLogger  # noqa: E402

cfg = config
wl_trace = TraceLogger()
wl_store = MemoryStore()
wl_client = client_from_env(use_cache=True)

b_loop = AgentLoop("agent_b", "NVDA", cfg.WHITELISTS["agent_b"], wl_client,
                   wl_trace, wl_store, cfg.ROLE_MAX_STEPS, "gather headlines only")
try:
    b_loop._dispatch(AgentAction(thought="peek at price",
                                 action=cfg.TOOL_GET_PRICE_DATA,
                                 args={"ticker": "NVDA", "period": "1y"}))
except CrossAgentToolAccessError as exc:
    print("REFUSED:", exc)

demo_blocked = demo_blocked_access("NVDA", wl_client, wl_trace, wl_store)
print("B continued with its own tools:",
      json.dumps(demo_blocked["continued_with"]))

REFUSED: tool 'get_price_data' is not permitted for agent_b (allowed: get_news, web_search)


B continued with its own tools: {"tool": "get_news", "ok": true, "headlines": 10}


## Part 6 - fault injection demo

`config.FAULT_INJECT = {"get_news": "empty"}` makes the news tool fail on
demand (the labelled test harness). The agent's observation log records the
failure and the loop stamps a replan before switching approach. AMD is used so
today's cache cannot shortcut the demo.

In [14]:
from src.agents import run_single_agent  # noqa: E402

memory_pkg.LAST_SESSION = memory_pkg.MemoryStore()  # isolate the session
cfg.FAULT_INJECT = {cfg.TOOL_GET_NEWS: "empty"}

fault_client = client_from_env(use_cache=True)
fault_trace = TraceLogger()
fault_store = MemoryStore()
fault_summary = run_single_agent("AMD", fault_client, fault_trace, fault_store)
print("composer degraded:", bool(fault_summary["report"].meta.degraded))

C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


C:\MIHA\CDAZZDEV\agentic_research\src\tools.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return list(DDGS().text(query, max_results=max_results))


single hit the step cap (8)


composer degraded: False


In [15]:
replans = fault_summary["replans"]
failed = [entry for entry in fault_summary["observations"] if not entry["ok"]]
print("failed (injected) calls:", [(e["tool"], e["digest"][:70]) for e in failed][:3])
print("stamped replans:")
for event in replans:
    print(" -", event["reason"][:110])
cfg.FAULT_INJECT = {}  # restore

failed (injected) calls: [('get_news', 'FAILED: injected fault (empty) (hint: this failure was injected by FAU'), ('get_news', 'FAILED: injected fault (empty) (hint: this failure was injected by FAU'), ('web_search', 'FAILED: no search results (hint: try a broader query)')]
stamped replans:
 - The first get_news attempt failed due to an injected fault; retrying to obtain the headlines required for llm_
 - get_news failed twice with injected faults; pivoting to web_search as the fallback source for market commentar
 - previous action 'get_news' failed (injected fault (empty)); switching approach
 - The first web_search failed with a hint to try a broader query, so I am broadening the search from a specific 
 - previous action 'web_search' failed (no search results); switching approach
 - get_news failed twice with injected faults and two narrower web_search queries returned no results; switching 
 - Three consecutive web_search failures and two get_news faults mean sentiment data is cur

## Part 7 - dashboard (run locally)

```bash
pip install streamlit plotly
streamlit run agentic_research/dashboards/trace_dashboard.py
```
Live screenshots are committed in `agentic_research/assets/dashboard.png`
(optional - see README).
